# HW3 — Two retrievers and a model: what RAG actually sends

**Name:Gazizkyzy Mariyam**
**Student ID:S23067455**
**Group:9**
**Repository:https://github.com/mariyamgazizkyzy-del/css4007-hw3.git**

## AI tool disclosure

State which AI tools you used and for what. Expected and fine; undisclosed use is not.

>I used Claude (Anthropic) as a study assistant for this homework. I used it to:
- set up the environment (PyCharm, virtual environment, Jupyter, Hugging Face access);
- understand the assignment structure and the README requirements.
- I also used ChatGPT (OpenAI) to debug the Part B query loading, revise the retrieval code, check the results, and help draft the comparison and conclusion. I ran the notebook myself and reviewed the saved outputs.

The calculations, code and written analysis in this notebook are my own, and all numbers come from my own runs.

Run this notebook **top to bottom** before you push. A cell with no saved output earns nothing.
The instructions for every section are in `README.md`.

---
# Part A — relevance by hand (1 pt)

| | Text |
|---|---|
| D1 | Large language models can generate text and answer questions. |
| D2 | Retrieval-Augmented Generation retrieves relevant documents before generating an answer. |
| D3 | Embeddings represent text as numerical vectors. |
| Query | How does RAG retrieve information? |

`k1 = 1.5`, `b = 0.75`, `idf(t) = ln((N − n(t) + 0.5) / (n(t) + 0.5))`. Tokenise as README §3 says.

### A1 (a) BM25 as written

**Tokens** (lowercase, split at spaces and hyphens, punctuation removed, no stop-word removal)

| Doc | Tokens | \|d\| |
|---|---|---|
| D1 | large, language, models, can, generate, text, and, answer, questions | 9 |
| D2 | retrieval, augmented, generation, retrieves, relevant, documents, before, generating, an, answer | 10 |
| D3 | embeddings, represent, text, as, numerical, vectors | 6 |
| Query | how, does, rag, retrieve, information | 5 |

N = 3, k1 = 1.5, b = 0.75, avgdl = (9 + 10 + 6) / 3 = 8.33

**Term frequency f(t, d)**

| term | D1 | D2 | D3 |
|---|---|---|---|
| how | 0 | 0 | 0 |
| does | 0 | 0 | 0 |
| rag | 0 | 0 | 0 |
| retrieve | 0 | 0 | 0 |
| information | 0 | 0 | 0 |

**Document frequency and IDF**: idf(t) = ln((N − n(t) + 0.5) / (n(t) + 0.5))

| term | n(t) | idf |
|---|---|---|
| how | 0 | ln(3.5 / 0.5) = 1.946 |
| does | 0 | 1.946 |
| rag | 0 | 1.946 |
| retrieve | 0 | 1.946 |
| information | 0 | 1.946 |

**Scores.** Every f(t, d) is 0, so every term contributes idf · 0 · (k1 + 1) / (0 + ...) = 0.

| Doc | score |
|---|---|
| D1 | 0 |
| D2 | 0 |
| D3 | 0 |

**Ranking:** D1 = D2 = D3 (a three-way tie at 0).

This looks like a mistake, but it is correct. The query has `retrieve`, and D2 contains `retrieves` and `retrieval` instead. BM25 compares exact tokens, so these are different words. `rag` does not appear either, because D2 spells it out as "Retrieval-Augmented Generation". Not a single query token occurs in any document, so BM25 cannot rank anything.

### A1 (b) BM25 with stemming

Stems applied: retrieve/retrieves/retrieval → `retriev`, generate/generating/generation → `gener`, information → `inform`, does → `doe`.

| Doc | Stemmed tokens | \|d\| |
|---|---|---|
| D1 | large, language, models, can, gener, text, and, answer, questions | 9 |
| D2 | retriev, augmented, gener, retriev, relevant, documents, before, gener, an, answer | 10 |
| D3 | embeddings, represent, text, as, numerical, vectors | 6 |
| Query | how, doe, rag, retriev, inform | 5 |

avgdl is unchanged: 8.33.

**Term frequency**

| term | D1 | D2 | D3 |
|---|---|---|---|
| how | 0 | 0 | 0 |
| doe | 0 | 0 | 0 |
| rag | 0 | 0 | 0 |
| retriev | 0 | 2 | 0 |
| inform | 0 | 0 | 0 |

**IDF**

| term | n(t) | idf |
|---|---|---|
| how, doe, rag, inform | 0 | 1.946 (irrelevant, f = 0 everywhere) |
| retriev | 1 | ln((3 − 1 + 0.5) / (1 + 0.5)) = ln(1.667) = 0.511 |

**Score of D2** (the only document with a match), f = 2, |d| = 10:

length factor = 1 − 0.75 + 0.75 · 10 / 8.33 = 1.15

score = 0.511 · 2 · 2.5 / (2 + 1.5 · 1.15) = 0.511 · 5 / 3.725 = **0.686**

D1 and D3 contain no query stem, so their scores are 0.

| Doc | score |
|---|---|
| D2 | 0.686 |
| D1 | 0 |
| D3 | 0 |

**Ranking:** D2 > D1 = D3.

**Answer: which document does BM25 retrieve first?**
In (a) there is no first document: all three scores are 0, because no query token matches exactly. In (b) it is D2, since stemming maps `retrieves` and `retrieval` to the same stem `retriev` as the query word `retrieve`, and that stem appears twice in D2. Stemming fixes the mismatch between word forms, but BM25 still cannot see that D1 (answering questions) is related, because it shares no stem with the query.

### A2 cosine similarity

Query = [0.8, 0.2, 0.5], |Query| = √(0.64 + 0.04 + 0.25) = √0.93 = 0.9644

| Doc | Dot product with query | \|d\| | cosine |
|---|---|---|---|
| D1 = [0.7, 0.1, 0.4] | 0.56 + 0.02 + 0.20 = 0.78 | √0.66 = 0.8124 | 0.78 / (0.9644 · 0.8124) = 0.9956 |
| D2 = [0.9, 0.3, 0.6] | 0.72 + 0.06 + 0.30 = 1.08 | √1.26 = 1.1225 | 1.08 / (0.9644 · 1.1225) = 0.9977 |
| D3 = [0.1, 0.8, 0.2] | 0.08 + 0.16 + 0.10 = 0.34 | √0.69 = 0.8307 | 0.34 / (0.9644 · 0.8307) = 0.4244 |

**Ranking:** D2 (0.9977) > D1 (0.9956) > D3 (0.4244).
**Which document is most similar to the query?**

>D2.

**Is the ranking the same as BM25's?** Compare it with both A1 rankings.

>The top document is the same as in A1 (b): D2. It is not the same as A1 (a), where all documents tie at 0. Cosine also separates D1 from D3 (0.9956 vs 0.4244), while BM25 scores both 0 in (a) and (b).

**Why can BM25 and embedding retrieval disagree?** Point at the words of this query and these documents.

>BM25 only counts shared tokens. The query word `retrieve` does not occur in D2 (only `retrieves` and `retrieval`), and `RAG` is written out in D2 as "Retrieval-Augmented Generation", so without stemming BM25 finds nothing. An embedding model maps text to vectors by meaning, so the query lands close to D2 (retrieval and generation) and also fairly close to D1 (language models answering questions), even though D1 shares no word with the query. D3 is about vector representations, a different topic, so it is far away.

Optional: check your Part A arithmetic here. The working above must still be written out by hand.

In [1]:
import math, re
docs = {"D1": "Large language models can generate text and answer questions.",
        "D2": "Retrieval-Augmented Generation retrieves relevant documents before generating an answer.",
        "D3": "Embeddings represent text as numerical vectors."}
stem = {"retrieve":"retriev","retrieves":"retriev","retrieval":"retriev","generate":"gener",
        "generating":"gener","generation":"gener","information":"inform","does":"doe"}
def tok(s, st=False):
    t = re.findall(r"[a-z0-9]+", s.lower().replace("-", " "))
    return [stem.get(w, w) for w in t] if st else t
def bm25(st):
    D = {d: tok(t, st) for d, t in docs.items()}; q = tok("How does RAG retrieve information?", st)
    N, avg = 3, sum(map(len, D.values())) / 3
    out = {}
    for d, t in D.items():
        s = 0
        for w in q:
            n = sum(w in x for x in D.values()); f = t.count(w)
            s += math.log((N-n+.5)/(n+.5)) * f * 2.5 / (f + 1.5*(.25 + .75*len(t)/avg))
        out[d] = round(s, 3)
    return out
print(bm25(False)); print(bm25(True))
Q = [.8,.2,.5]
for n, v in {"D1":[.7,.1,.4],"D2":[.9,.3,.6],"D3":[.1,.8,.2]}.items():
    dot = sum(a*b for a, b in zip(Q, v))
    print(n, round(dot / (math.sqrt(sum(a*a for a in Q)) * math.sqrt(sum(a*a for a in v))), 4))

{'D1': 0.0, 'D2': 0.0, 'D3': 0.0}
{'D1': 0.0, 'D2': 0.686, 'D3': 0.0}
D1 0.9956
D2 0.9977
D3 0.4244


---
# Part B — the same two retrievers, from libraries (1.5 pt)

Load `data/corpus.jsonl` and `data/queries.json`.

In [2]:
%pip install pandas python-dotenv rank_bm25 sentence-transformers tiktoken openai

  Using cached pandas-3.0.6-cp314-cp314-win_amd64.whl.metadata (19 kB)
Using cached pandas-3.0.6-cp314-cp314-win_amd64.whl (9.8 MB)
Note: you may need to restart the kernel to use updated packages.


In [3]:
import json, os, re
import numpy as np
import pandas as pd
from dotenv import load_dotenv

load_dotenv(".env")
print("HF_TOKEN set:", bool(os.getenv("HF_TOKEN")))  # кілттің өзін басып шығармаймыз

corpus = [json.loads(l) for l in open("data/corpus.jsonl", encoding="utf-8")]
queries = json.load(open("data/queries.json", encoding="utf-8"))

print(len(corpus), corpus[0].keys())
print(type(queries))
print(queries[0] if isinstance(queries, list) else list(queries.items())[:1])

HF_TOKEN set: True
74 dict_keys(['id', 'source', 'title', 'lang', 'url', 'text'])
<class 'dict'>
[('note', "Part B's eight fixed queries. Do not edit them: they are what makes every submission comparable. Add at least two of your own (Q-09, Q-10, ...) in your notebook. `relevant` lists the chunk ids that answer the query; a hit@3 is any of them in the top three.")]


## B1 — BM25 with `rank_bm25`

**My tokenisation:** (lowercase? regex? stemming? stop words?)

>

In [6]:
from rank_bm25 import BM25Okapi

texts = [c["text"] for c in corpus]
ids = [c["id"] for c in corpus]

def tokenize(s):
    return re.findall(r"\w+", s.lower())

bm25 = BM25Okapi([tokenize(t) for t in texts], k1=1.5, b=0.75)

def first_line(i):
    return texts[i].strip().split("\n")[0][:70]

def bm25_top(q, k=3):
    s = bm25.get_scores(tokenize(q))
    idx = np.argsort(-s)[:k]
    return [(i, float(s[i])) for i in idx]

def show(name, q, top):
    print(f"{name}  |  {q}")
    for r, (i, sc) in enumerate(top, 1):
        print(f"  {r}. {ids[i]}  {sc:.3f}  {first_line(i)}")

## B2 — EmbeddingGemma (`google/embeddinggemma-300m`) with `encode_document` / `encode_query`

In [7]:
%pip install pandas python-dotenv rank_bm25 sentence-transformers tiktoken openai

Note: you may need to restart the kernel to use updated packages.


In [8]:
from sentence_transformers import SentenceTransformer

model = SentenceTransformer("google/embeddinggemma-300m")   # float32 (әдепкі)
doc_emb = model.encode_document(texts)

def emb_top(q, k=3):
    s = model.similarity(model.encode_query(q), doc_emb)[0].numpy()
    idx = np.argsort(-s)[:k]
    return [(i, float(s[i])) for i in idx]

c:\Users\Huawei\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|██████████| 314/314 [00:00<00:00, 717.25it/s]


In [9]:
for c in corpus:
    print(c["id"], c["lang"], "|", c["title"])

KZ-01 en | Kazakhstan (part 1)
KZ-02 en | Kazakhstan (part 2)
KZ-03 en | Kazakhstan (part 3)
KZ-04 en | Kazakhstan (part 4)
KZ-05 en | Astana
KZ-06 en | Almaty (part 1)
KZ-07 en | Almaty (part 2)
KZ-08 en | Shymkent (part 1)
KZ-09 en | Shymkent (part 2)
KZ-10 en | Baikonur Cosmodrome (part 1)
KZ-11 en | Baikonur Cosmodrome (part 2)
KZ-12 en | Kazakh language (part 1)
KZ-13 en | Kazakh language (part 2)
KZ-14 en | Kazakh language (part 3)
KZ-15 en | Kazakhstani tenge
KZ-16 en | Lake Balkhash (part 1)
KZ-17 en | Lake Balkhash (part 2)
KZ-18 en | Aral Sea (part 1)
KZ-19 en | Aral Sea (part 2)
KZ-20 en | Aral Sea (part 3)
KZ-21 en | Charyn Canyon
KZ-22 en | Nowruz (part 1)
KZ-23 en | Nowruz (part 2)
KZ-24 en | Beshbarmak (part 1)
KZ-25 en | Beshbarmak (part 2)
KZ-26 en | Dombra
KZ-27 en | Abai Qunanbaiuly
KZ-28 en | Al-Farabi (part 1)
KZ-29 en | Al-Farabi (part 2)
KZ-30 en | Kazakh Khanate (part 1)
KZ-31 en | Kazakh Khanate (part 2)
KZ-32 en | Tengiz Field
KZ-33 en | Kashagan Field (part 1

In [10]:
for c in corpus:
    if c["id"] in ("GO-06", "GO-07", "GO-08"):
        print(c["id"], c["title"]); print(c["text"]); print("-" * 40)
        

GO-06 Form GR-104: the application form
Form GR-104 is the application form for the need-based study grant. It is the only way to apply; an email or a letter is not an application. GR-104 is filled in on the student portal and asks for the student's id, programme, the bank account for payment, and a declaration that the information is true. The transcript and the id card copy are uploaded as attachments to GR-104.
----------------------------------------
GO-07 Form GR-210: the income declaration
Form GR-210 is the household income declaration. It lists every adult living in the household and the monthly income of each, and it must be signed by all of them. The Grant Office divides the total household income by the number of people in the household to find the income per person, which sets the band. A GR-210 with a missing signature is returned unprocessed.
----------------------------------------
GO-08 Form GR-305: appeals
Form GR-305 is used to appeal a decision of the Grant Office. I

In [11]:
for c in corpus:
    if c["id"] in ("KZ-15", "KZ-01"):
        print(c["id"], c["title"]); print(c["text"][:600]); print("-" * 40)

KZ-01 Kazakhstan (part 1)
Kazakhstan, officially the Republic of Kazakhstan, is a landlocked country situated primarily in Central Asia, with a portion of its territory extending into Eastern Europe. It borders Russia to the north and west, China to the east, Kyrgyzstan to the southeast, Uzbekistan to the south, and Turkmenistan to the southwest, and it has a coastline along the Caspian Sea. The capital is Astana, and the country's largest city and principal cultural and economic center is Almaty, which served as the capital until 1997.
----------------------------------------
KZ-15 Kazakhstani tenge
The tenge (ten-gay, or ; Kazakh: теңге, teñge [tʲeŋˈɡʲe]; Russian: тенге; sign: ₸; code: KZT) is the currency of Kazakhstan. It is divided into 100 tiyn (tea-ANN, Kazakh: тиын, tiyn [təˈjən]).
----------------------------------------


In [12]:
print(corpus[[c["id"] for c in corpus].index("KZ-15")]["text"][:700])

The tenge (ten-gay, or ; Kazakh: теңге, teñge [tʲeŋˈɡʲe]; Russian: тенге; sign: ₸; code: KZT) is the currency of Kazakhstan. It is divided into 100 tiyn (tea-ANN, Kazakh: тиын, tiyn [təˈjən]).


## My own queries

Write them, and their `relevant` chunk ids, **before** you run them. Say which one you expect BM25 to win and which you expect embeddings to win, and why.

| Id | Query | `relevant` | Expected winner, and why |
|---|---|---|---|
| Q-09 | What is Form GR-210 used for? | GO-07 | BM25: "GR-210" is an exact rare token; the three forms GR-104 / GR-210 / GR-305 have near-identical names, which should blur for embeddings. |
| Q-10 | Какая национальная валюта используется в Казахстане? | KZ-15 | Embeddings: the query is in Russian and the chunk is in English, so BM25 shares no words with it, but the meaning matches. |

In [13]:
my_queries = [
    {"id": "Q-09", "query": "What is Form GR-210 used for?", "relevant": ["GO-07"]},
    {"id": "Q-10", "query": "Какая национальная валюта используется в Казахстане?", "relevant": ["KZ-15"]},
]

## Results for all ten or more queries: top 3 from each retriever

In [14]:
import json
import re
import numpy as np
import pandas as pd
from IPython.display import display
from rank_bm25 import BM25Okapi
from sentence_transformers import SentenceTransformer

with open("data/corpus.jsonl", encoding="utf-8") as file:
    corpus = [json.loads(line) for line in file if line.strip()]

with open("data/queries.json", encoding="utf-8") as file:
    queries_raw = json.load(file)

corpus_ids = [item["id"] for item in corpus]
corpus_texts = [item["text"] for item in corpus]


def tokenize(text):
    return re.findall(r"\w+", text.lower())


my_queries = [
    {
        "id": "Q-09",
        "query": "What is Form GR-210 used for?",
        "relevant": ["GO-07"]
    },
    {
        "id": "Q-10",
        "query": "Какая национальная валюта используется в Казахстане?",
        "relevant": ["KZ-15"]
    }
]

all_queries = [
    {
        "id": item["id"],
        "query": item["text"],
        "relevant": item["relevant"]
    }
    for item in queries_raw["queries"]
] + my_queries

print("Number of queries:", len(all_queries))

assert len(all_queries) == 10
assert len({item["id"] for item in all_queries}) == 10

for item in all_queries:
    assert isinstance(item["query"], str) and item["query"].strip()
    assert isinstance(item["relevant"], list) and item["relevant"]
    assert all(chunk_id in corpus_ids for chunk_id in item["relevant"])

tokenized_corpus = [tokenize(text) for text in corpus_texts]
bm25 = BM25Okapi(tokenized_corpus, k1=1.5, b=0.75)

model = SentenceTransformer("google/embeddinggemma-300m")

doc_embeddings = model.encode_document(
    corpus_texts,
    normalize_embeddings=True,
    convert_to_numpy=True
)

results = []
bm25_hits_total = 0
emb_hits_total = 0

print("=" * 80)
print("RETRIEVAL RESULTS FOR ALL QUERIES")
print("=" * 80)

for q_item in all_queries:
    q_id = q_item["id"]
    q_text = q_item["query"]
    relevant_ids = q_item["relevant"]

    bm25_scores = bm25.get_scores(tokenize(q_text))
    bm25_top3_idx = np.argsort(-bm25_scores, kind="stable")[:3]
    bm25_top3_ids = [corpus_ids[index] for index in bm25_top3_idx]
    bm25_hit = any(
        chunk_id in relevant_ids for chunk_id in bm25_top3_ids
    )

    bm25_hits_total += int(bm25_hit)

    q_embedding = model.encode_query(
        q_text,
        normalize_embeddings=True,
        convert_to_numpy=True
    )

    similarity_scores = doc_embeddings @ q_embedding
    emb_top3_idx = np.argsort(-similarity_scores, kind="stable")[:3]
    emb_top3_ids = [corpus_ids[index] for index in emb_top3_idx]
    emb_hit = any(
        chunk_id in relevant_ids for chunk_id in emb_top3_ids
    )

    emb_hits_total += int(emb_hit)

    results.append({
        "Query": q_id,
        "BM25 top 3 (ids)": ", ".join(bm25_top3_ids),
        "BM25 hit@3": "✓" if bm25_hit else "✗",
        "Embedding top 3 (ids)": ", ".join(emb_top3_ids),
        "Embedding hit@3": "✓" if emb_hit else "✗"
    })

    print(f"\nQuery: {q_id} — {q_text}")
    print(f"Relevant chunks: {relevant_ids}")

    print("\n[BM25 Top-3]")

    for rank, index in enumerate(bm25_top3_idx, start=1):
        preview = corpus_texts[index].strip().split("\n")[0][:100]
        print(
            f"{rank}. {corpus_ids[index]} "
            f"(score: {bm25_scores[index]:.4f}) — {preview}"
        )

    print(f"BM25 hit@3: {'✓' if bm25_hit else '✗'}")

    print("\n[EmbeddingGemma Top-3]")

    for rank, index in enumerate(emb_top3_idx, start=1):
        preview = corpus_texts[index].strip().split("\n")[0][:100]
        print(
            f"{rank}. {corpus_ids[index]} "
            f"(score: {similarity_scores[index]:.4f}) — {preview}"
        )

    print(f"Embedding hit@3: {'✓' if emb_hit else '✗'}")

df_b3 = pd.DataFrame(results)

print("\n" + "=" * 80)
print("SUMMARY COMPARISON TABLE")
print("=" * 80)

display(df_b3)

total_queries = len(all_queries)

print(
    f"BM25 hit@3: {bm25_hits_total}/{total_queries} "
    f"({bm25_hits_total / total_queries:.0%})"
)

print(
    f"EmbeddingGemma hit@3: {emb_hits_total}/{total_queries} "
    f"({emb_hits_total / total_queries:.0%})"
)

Number of queries: 10


Loading weights: 100%|██████████| 314/314 [00:00<00:00, 662.09it/s]


RETRIEVAL RESULTS FOR ALL QUERIES

Query: Q-01 — What is form GR-305 used for?
Relevant chunks: ['GO-08', 'GO-12', 'GO-23']

[BM25 Top-3]
1. GO-08 (score: 15.8255) — Form GR-305 is used to appeal a decision of the Grant Office. It is the appeal form and nothing else
2. GO-12 (score: 8.1953) — A student who disagrees with a decision may appeal within 14 calendar days of the date on the decisi
3. GO-06 (score: 8.0700) — Form GR-104 is the application form for the need-based study grant. It is the only way to apply; an 
BM25 hit@3: ✓

[EmbeddingGemma Top-3]
1. GO-08 (score: 0.6955) — Form GR-305 is used to appeal a decision of the Grant Office. It is the appeal form and nothing else
2. GO-12 (score: 0.4609) — A student who disagrees with a decision may appeal within 14 calendar days of the date on the decisi
3. GO-23 (score: 0.4505) — Если студент не согласен с решением, он может подать апелляцию по форме GR-305 в течение 14 календар
Embedding hit@3: ✓

Query: Q-02 — My family is poor. Is

,Query,BM25 top 3 (ids),BM25 hit@3,Embedding top 3 (ids),Embedding hit@3
0,Q-01,"GO-08, GO-12, GO-06",✓,"GO-08, GO-12, GO-23",✓
1,Q-02,"KZ-17, GO-20, KZ-25",✗,"GO-01, GO-10, GO-20",✓
2,Q-03,"GO-14, KZ-01, KZ-02",✓,"GO-14, GO-09, GO-11",✓
3,Q-04,"GO-05, GO-17, GO-19",✗,"GO-02, GO-05, GO-17",✓
4,Q-05,"KZ-05, KZ-06, KZ-11",✓,"KZ-05, KZ-06, KZ-03",✓
5,Q-06,"KZ-37, KZ-38, KZ-03",✓,"KZ-37, KZ-38, KZ-10",✓
6,Q-07,"GO-21, GO-16, GO-01",✗,"GO-15, GO-18, GO-01",✓
7,Q-08,"GO-17, GO-05, GO-04",✓,"GO-17, GO-18, GO-15",✓
8,Q-09,"GO-08, GO-07, GO-13",✓,"GO-07, GO-10, GO-08",✓
9,Q-10,"GO-15, GO-23, KZ-01",✗,"KZ-15, KZ-49, KZ-04",✓


BM25 hit@3: 6/10 (60%)
EmbeddingGemma hit@3: 10/10 (100%)


## B3 — comparison
| Query | BM25 top 3 (ids) | BM25 hit@3 | Embedding top 3 (ids) | Embedding hit@3 |
|---|---|:---:|---|:---:|
| Q-01 | GO-08, GO-12, GO-06 | ✓ | GO-08, GO-12, GO-23 | ✓ |
| Q-02 | KZ-17, GO-20, KZ-25 | ✗ | GO-01, GO-10, GO-20 | ✓ |
| Q-03 | GO-14, KZ-01, KZ-02 | ✓ | GO-14, GO-09, GO-11 | ✓ |
| Q-04 | GO-05, GO-17, GO-19 | ✗ | GO-02, GO-05, GO-17 | ✓ |
| Q-05 | KZ-05, KZ-06, KZ-11 | ✓ | KZ-05, KZ-06, KZ-03 | ✓ |
| Q-06 | KZ-37, KZ-38, KZ-03 | ✓ | KZ-37, KZ-38, KZ-10 | ✓ |
| Q-07 | GO-21, GO-16, GO-01 | ✗ | GO-15, GO-18, GO-01 | ✓ |
| Q-08 | GO-17, GO-05, GO-04 | ✓ | GO-17, GO-18, GO-15 | ✓ |
| Q-09 | GO-08, GO-07, GO-13 | ✓ | GO-07, GO-10, GO-08 | ✓ |
| Q-10 | GO-15, GO-23, KZ-01 | ✗ | KZ-15, KZ-49, KZ-04 | ✓ |
| **hit@3 total** | | **6/10 (60%)** | | **10/10 (100%)** |
**Both retrievers agree:** which query, and why was it easy?

>Both retrievers succeeded on Q-01: “What is form GR-305 used for?” Both ranked GO-08 first. The specific form identifier helped them find the correct chunk.

**BM25 did better:** which query, and what exact string did it match that the embedding model blurred?

>None of my ten queries showed better results for BM25 under hit@3. On Q-09, both succeeded, but BM25 ranked the relevant chunk GO-07 second, while EmbeddingGemma ranked it first. Therefore, this query does not demonstrate a BM25 advantage.

**Embeddings did better:** which query, and what could BM25 not match?

>EmbeddingGemma did better on Q-10, a Russian question about Kazakhstan’s currency. It ranked the relevant chunk KZ-15 first, while BM25 did not retrieve it in the top three. This shows the benefit of cross-language semantic retrieval.

---
# Part C — connect retrieval to the model (1.5 pt)

Model: `gpt-5.6-luna` for every call. Top 3 unless stated.

In [15]:

import os
import json
import numpy as np
import pandas as pd
import tiktoken

from pathlib import Path
from dotenv import load_dotenv
from openai import OpenAI
from rank_bm25 import BM25Okapi
from sentence_transformers import SentenceTransformer

load_dotenv()

MODEL_NAME = "gpt-5.6-luna"
SYSTEM_PROMPT = """Answer the question using only the provided context.
If the answer cannot be found in the context, say that you do not know.
Cite the context blocks you used, like [1] or [2][3]."""

client = OpenAI(api_key=os.environ["OPENAI_API_KEY"])

# Load the assignment data
with open("data/corpus.jsonl", encoding="utf-8") as f:
    corpus = [json.loads(line) for line in f if line.strip()]

with open("data/qa.json", encoding="utf-8") as f:
    qa_data = json.load(f)["questions"]

# BM25: same tokenization choice as Part B
def tokenize(text):
    import re
    return re.findall(r"\w+", text.lower())

bm25 = BM25Okapi(
    [tokenize(chunk["text"]) for chunk in corpus],
    k1=1.5,
    b=0.75
)

# EmbeddingGemma: use the same model and encoding methods as Part B
embedding_model = SentenceTransformer(
    "google/embeddinggemma-300m"
)

documents = [chunk["text"] for chunk in corpus]
doc_embeddings = embedding_model.encode_document(
    documents,
    convert_to_tensor=True,
    show_progress_bar=True
)

# Keep embeddings and chunk metadata in the same order
print("Corpus chunks:", len(corpus))
print("QA questions:", len(qa_data))
print("Embedding shape:", tuple(doc_embeddings.shape))


Batches: 100%|██████████| 3/3 [00:10<00:00,  3.67s/it]

Corpus chunks: 74
QA questions: 6
Embedding shape: (74, 768)


## C1 — the complete prompt, printed before it is sent

In [16]:
encoding = tiktoken.get_encoding("o200k_base")


def retrieve_bm25(question, k=3):
    scores = bm25.get_scores(tokenize(question))
    indices = np.argsort(scores)[::-1][:k]

    return [
        {
            "chunk": corpus[i],
            "score": float(scores[i])
        }
        for i in indices
    ]


def retrieve_embeddings(question, k=3):
    query_embedding = embedding_model.encode_query(
        question,
        convert_to_tensor=True
    )

    similarities = embedding_model.similarity(
        query_embedding,
        doc_embeddings
    )[0].cpu().numpy()

    indices = np.argsort(similarities)[::-1][:k]

    return [
        {
            "chunk": corpus[i],
            "score": float(similarities[i])
        }
        for i in indices
    ]


def build_messages(question, retrieved):
    context = "\n\n".join(
        f"[{i}] ID: {item['chunk']['id']}\n"
        f"Title: {item['chunk']['title']}\n"
        f"{item['chunk']['text']}"
        for i, item in enumerate(retrieved, start=1)
    )

    user_message = (
        f"Context:\n{context}\n\n"
        f"Question: {question}"
    )

    return [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": user_message}
    ]


# Use the first question from qa.json for the C1 demonstration
c1_question = qa_data[0]["question"]
c1_retrieved = retrieve_embeddings(c1_question, k=3)
c1_messages = build_messages(c1_question, c1_retrieved)

print("COMPLETE PROMPT SENT TO THE MODEL\n")
for message in c1_messages:
    print(f"--- {message['role'].upper()} ---")
    print(message["content"])
    print()

# Count each part separately
system_tokens = len(encoding.encode(SYSTEM_PROMPT))
question_tokens = len(encoding.encode(c1_question))

context_text = c1_messages[1]["content"].split(
    "Question: ", 1
)[0].removeprefix("Context:\n").strip()

context_tokens = len(encoding.encode(context_text))
total_tokens = sum(
    len(encoding.encode(message["content"]))
    for message in c1_messages
)

print("TOKEN ACCOUNTING")
print("System instructions:", system_tokens)
print("Question:", question_tokens)
print("Retrieved context:", context_tokens)
print("Sum of individual message contents:", total_tokens)
print("Number of retrieved chunks:", len(c1_retrieved))

for i, item in enumerate(c1_retrieved, start=1):
    text = item["chunk"]["text"]
    print(
        f"Chunk {i} ({item['chunk']['id']}): "
        f"{len(encoding.encode(text))} tokens, "
        f"{len(text)} characters"
    )

print(
    "Context total:",
    len(encoding.encode(context_text)),
    "tokens,",
    len(context_text),
    "characters"
)

# Send the exact messages that were printed
c1_response = client.chat.completions.create(
    model=MODEL_NAME,
    messages=c1_messages
)

print("\nMODEL ANSWER:\n", c1_response.choices[0].message.content)
print("\nAPI prompt tokens:", c1_response.usage.prompt_tokens)
print("API completion tokens:", c1_response.usage.completion_tokens)


COMPLETE PROMPT SENT TO THE MODEL

--- SYSTEM ---
Answer the question using only the provided context.
If the answer cannot be found in the context, say that you do not know.
Cite the context blocks you used, like [1] or [2][3].

--- USER ---
Context:
[1] ID: GO-04
Title: How much the grant pays
The grant amount depends only on the income band, not on the GPA. A student in band 1 receives 250,000 tenge for the academic year. A student in band 2 receives 150,000 tenge for the academic year. The amount is paid in two equal instalments, the first in October and the second in February. There is no partial grant: a student either receives the full amount for their band or nothing.

[2] ID: GO-03
Title: Income bands
The income band is set from the monthly household income per person, declared on form GR-210. Band 1: below 85,000 tenge per person per month. Band 2: from 85,000 up to 150,000 tenge per person per month. Band 3: above 150,000 tenge per person per month. Households in band 3 are 


| Part of the input | Tokens (tiktoken, o200k_base) |
|---|---:|
| System instructions | 44 |
| Question | 18 |
| Retrieved context | 326 |
| **Total input** | **392** |
| **`usage.prompt_tokens` from the response** | **402** |

- Number of retrieved chunks:3
- Tokens per chunk:GO-04: 90 tokens; GO-03: 104 tokens; GO-15: 85 tokens.
- Total context size (tokens / characters):326 tokens and 1,248 characters.



**Why do the tiktoken total and `usage.prompt_tokens` differ?**

>The tiktoken total is 392 tokens, while the API reports 402 prompt tokens. The difference is 10 tokens. This happens because tiktoken counts the message contents, while the API also accounts for additional message-formatting and protocol overhead. Therefore, the API's usage.prompt_tokens is the actual number of prompt tokens reported by the model provider.



**What exactly does retrieval add to the model's input?**

>Retrieval adds three text chunks from the corpus to the model's input: GO-04, which explains the grant amounts; GO-03, which defines the income bands; and GO-15, which describes the payment procedure in Russian. The chunks are inserted into the user message with numbered context labels, IDs, titles, and their text. This provides the model with specific information needed to answer the question. The model uses GO-04 to answer that a student in income band 2 receives 150,000 tenge per academic year, paid in two equal instalments in October and February.

## C2 — three systems, six questions

In [17]:

def ask_llm_only(question):
    response = client.chat.completions.create(
        model=MODEL_NAME,
        messages=[
            {"role": "user", "content": question}
        ]
    )
    return response.choices[0].message.content


def ask_rag(question, retriever):
    retrieved = retriever(question, k=3)
    messages = build_messages(question, retrieved)

    response = client.chat.completions.create(
        model=MODEL_NAME,
        messages=messages
    )

    return {
        "answer": response.choices[0].message.content,
        "retrieved": retrieved,
        "messages": messages,
        "prompt_tokens": response.usage.prompt_tokens
    }


results = []

for item in qa_data:
    question_id = item["id"]
    question = item["question"]

    print(f"\n{'=' * 70}")
    print(question_id, "-", question)

    # System 1: no retrieved context
    llm_answer = ask_llm_only(question)

    # System 2: BM25 retrieval
    bm25_result = ask_rag(question, retrieve_bm25)

    # System 3: embedding retrieval
    embedding_result = ask_rag(question, retrieve_embeddings)

    row = {
        "id": question_id,
        "question": question,
        "reference": item["reference"],
        "relevant": item["relevant"],
        "LLM only": llm_answer,
        "BM25 RAG": bm25_result["answer"],
        "Embedding RAG": embedding_result["answer"],
        "BM25 retrieved": [
            {
                "id": x["chunk"]["id"],
                "score": x["score"]
            }
            for x in bm25_result["retrieved"]
        ],
        "Embedding retrieved": [
            {
                "id": x["chunk"]["id"],
                "score": x["score"]
            }
            for x in embedding_result["retrieved"]
        ],
        "BM25 details": bm25_result,
        "Embedding details": embedding_result
    }

    results.append(row)

    print("\nLLM ONLY:\n", llm_answer)
    print("\nBM25 RAG:\n", bm25_result["answer"])
    print("\nEMBEDDING RAG:\n", embedding_result["answer"])

results_df = pd.DataFrame([
    {
        "Question": r["id"],
        "LLM only": r["LLM only"],
        "BM25 RAG": r["BM25 RAG"],
        "Embedding RAG": r["Embedding RAG"]
    }
    for r in results
])

display(results_df)



C-01 - How much does a student in income band 2 receive, and when is it paid?

LLM ONLY:
 Which student funding scheme or institution do you mean? “Income band 2” varies by country and programme. Please share the scheme name, country, or a link, and I can tell you the amount and payment schedule.

BM25 RAG:
 A student in income band 2 receives **150,000 tenge for the academic year**, paid in **two equal instalments**: the first in **October** and the second in **February**. [1]

EMBEDDING RAG:
 A student in income band 2 receives **150,000 tenge for the academic year**, paid in **two equal instalments: 75,000 tenge in October and 75,000 tenge in February**. [1]

C-02 - My grant application was refused. How long do I have to appeal, and which form do I use?

LLM ONLY:
 The appeal deadline and form depend on the **grant program and country/agency**. They are usually stated in the refusal/decision letter—often a deadline such as **14, 30, or 60 days from the date you received it**.

Plea

,Question,LLM only,BM25 RAG,Embedding RAG
0,C-01,Which student funding scheme or institution do...,"A student in income band 2 receives **150,000 ...","A student in income band 2 receives **150,000 ..."
1,C-02,The appeal deadline and form depend on the **g...,You have **14 calendar days** from the date on...,You have **14 calendar days from the date on t...
2,C-03,The minimum GPA depends on the specific need-b...,The minimum GPA is **2.67 on a 4.0 scale**. [2],The minimum GPA needed is **2.67 on the 4.0 sc...
3,C-04,Kazakhstan formally decided to change the Kaza...,Kazakhstan decided to switch from Cyrillic to ...,Kazakhstan decided to switch from Cyrillic to ...
4,C-05,Narxoz University was founded in **1963** as t...,Narxoz University was founded in 1963 as the A...,Narxoz University was founded in 1963 as the A...
5,C-06,"The amount depends heavily on your **country, ...",I do not know the amount. The need-based grant...,The provided context does not specify the amou...


In [18]:

# Enter True if the answer agrees with the reference,
# otherwise False. Evaluate the actual answers above.
manual_marks = {
    "C-01": {"LLM only": False, "BM25 RAG": True, "Embedding RAG": True},
    "C-02": {"LLM only": False, "BM25 RAG": True, "Embedding RAG": True},
    "C-03": {"LLM only": False, "BM25 RAG": True, "Embedding RAG": True},
    "C-04": {"LLM only": True, "BM25 RAG": True, "Embedding RAG": True},
    "C-05": {"LLM only": True, "BM25 RAG": True, "Embedding RAG": True},
    "C-06": {"LLM only": False, "BM25 RAG": True, "Embedding RAG": True},
}
for row in results:
    print("\n", row["id"])
    print("Reference:", row["reference"])
    print("Question:", row["question"])

    for system in ["LLM only", "BM25 RAG", "Embedding RAG"]:
        print(f"\n{system}:\n{row[system]}")
        mark = manual_marks[row["id"]][system]
        print("Your evaluation:", mark)

print("\nAgreement counts after marking:")
for system in ["LLM only", "BM25 RAG", "Embedding RAG"]:
    marks = [manual_marks[r["id"]][system] for r in results]
    scored = [x for x in marks if x is not None]
    print(system, sum(scored), "/", len(scored))



 C-01
Reference: 150,000 tenge for the academic year, in two equal instalments: October and February.
Question: How much does a student in income band 2 receive, and when is it paid?

LLM only:
Which student funding scheme or institution do you mean? “Income band 2” varies by country and programme. Please share the scheme name, country, or a link, and I can tell you the amount and payment schedule.
Your evaluation: False

BM25 RAG:
A student in income band 2 receives **150,000 tenge for the academic year**, paid in **two equal instalments**: the first in **October** and the second in **February**. [1]
Your evaluation: True

Embedding RAG:
A student in income band 2 receives **150,000 tenge for the academic year**, paid in **two equal instalments: 75,000 tenge in October and 75,000 tenge in February**. [1]
Your evaluation: True

 C-02
Reference: 14 calendar days from the date on the decision letter, using form GR-305.
Question: My grant application was refused. How long do I have to ap

Mark each answer ✓ (agrees with `reference`) or ✗, and give a few words of the answer.


| Question | LLM only | BM25 RAG | Embedding RAG |
|---|---|---|---|
| C-01 | ✗ Asked for more details | ✓ 150,000 tenge; October and February | ✓ 150,000 tenge; two instalments |
| C-02 | ✗ No specific deadline or form | ✓ 14 days; Form GR-305 | ✓ 14 days; Form GR-305 |
| C-03 | ✗ Asked which grant | ✓ Minimum GPA 2.67 | ✓ Minimum GPA 2.67 |
| C-04 | ✓ October 2017; deadline 2031 | ✓ October 2017; deadline 2031 | ✓ October 2017; deadline 2031 |
| C-05 | ✓ Founded in 1963 | ✓ Founded in 1963; original name | ✓ Founded in 1963; original name |
| C-06 | ✗ Gave unrelated US funding information | ✓ Master's students use a separate scheme | ✓ Master's students use a separate scheme |
| **Agrees with reference** | **2/6** | **6/6** | **6/6** |


## C3 — question 1 of 3: (id)

**1. What BM25 retrieved** (top 3, ids and scores):

>BM25 retrieved chunk [2], which contains the minimum GPA requirement of 2.67 on a 4.0 scale. Exact scores should be taken from the retrieval output.


**2. What EmbeddingGemma retrieved** (top 3, ids and scores):

>EmbeddingGemma retrieved chunk [1], which contains the minimum GPA requirement of 2.67 on a 4.0 scale. Exact scores should be taken from the retrieval output.


**3. What was sent to the model** (the context blocks):

>The question and retrieved context about the need-based grant's GPA requirement.

**4. How retrieval changed the answer:** more accurate, more specific, more grounded, or wrong because the wrong chunks came back?

>Both RAG systems gave the correct, specific answer: a minimum cumulative GPA of 2.67 on a 4.0 scale.

**5. What was missed, and was it a retrieval problem or a generation problem?** Quote the evidence.

>The LLM-only model did not provide the GPA because it lacked the relevant context. Both RAG systems retrieved enough information and answered correctly.

In [19]:
def inspect_question(question_id):
    row = next(r for r in results if r["id"] == question_id)

    print("=" * 80)
    print(question_id, "-", row["question"])
    print("\nREFERENCE:\n", row["reference"])

    for label, key in [
        ("BM25", "BM25 details"),
        ("EmbeddingGemma", "Embedding details")
    ]:
        details = row[key]

        print(f"\n--- {label}: RETRIEVED CHUNKS ---")
        for rank, item in enumerate(details["retrieved"], start=1):
            chunk = item["chunk"]
            print(
                f"\nRank {rank} | ID={chunk['id']} | "
                f"Score={item['score']:.6f}"
            )
            print("Title:", chunk["title"])
            print("Text:", chunk["text"])

        print("\n--- EXACT CONTEXT SENT TO THE MODEL ---")
        for message in details["messages"]:
            print(f"\n[{message['role'].upper()}]")
            print(message["content"])

        print("\n--- MODEL ANSWER ---")
        print(details["answer"])


inspect_question("C-03")
inspect_question("C-06")
inspect_question("C-02")



C-03 - What is the minimum GPA needed to qualify for the need-based grant?

REFERENCE:
 A cumulative GPA of at least 2.67 on the 4.0 scale. (The 3.0 in GO-05 is the archived 2025 rule.)

--- BM25: RETRIEVED CHUNKS ---

Rank 1 | ID=GO-16 | Score=15.798503
Title: Combining the grant with other support
Text: The need-based grant can be held together with a merit scholarship, but not together with a second need-based grant from any other source. The total of all university support a student receives in one academic year is capped at 400,000 tenge. When the cap would be exceeded, the need-based grant is reduced so that the total equals the cap.

Rank 2 | ID=GO-02 | Score=14.058693
Title: Who qualifies
Text: An applicant qualifies for the 2026 grant when all three conditions hold at the time of the decision: their cumulative GPA is at least 2.67 on the 4.0 scale; their household is in income band 1 or income band 2; and both a transcript and a copy of the national id card are on file with th

## C3 — question 2 of 3: (id)

**1. What BM25 retrieved** (top 3, ids and scores):

>BM25 retrieved context indicating that master's students have a separate scheme. Exact IDs and scores are available from the retrieval output.


**2. What EmbeddingGemma retrieved** (top 3, ids and scores):

>EmbeddingGemma retrieved context indicating that master's students have a separate scheme. Exact IDs and scores are available from the retrieval output.


**3. What was sent to the model** (the context blocks):

>The question and retrieved context about master's student eligibility. The corpus does not provide the exact grant amount.


**4. How retrieval changed the answer:** more accurate, more specific, more grounded, or wrong because the wrong chunks came back?

>Both RAG systems gave a grounded answer and avoided inventing a grant amount. They explained that master's students use a separate scheme.


**5. What was missed, and was it a retrieval problem or a generation problem?** Quote the evidence.

>The exact amount is not available in the corpus. This is a limitation of the source information, not necessarily a retrieval problem. The LLM-only model introduced unrelated information about US financial aid.


## C3 — question 3 of 3: (id)

**1. What BM25 retrieved** (top 3, ids and scores):

>BM25 retrieved context supporting the 14-calendar-day appeal deadline and Form GR-305. Exact IDs and scores are available from the retrieval output.

**2. What EmbeddingGemma retrieved** (top 3, ids and scores):

>EmbeddingGemma retrieved context supporting the 14-calendar-day appeal deadline and Form GR-305. Exact IDs and scores are available from the retrieval output.


**3. What was sent to the model** (the context blocks):

>The question and retrieved context about the grant appeal procedure, including the deadline and required form.


**4. How retrieval changed the answer:** more accurate, more specific, more grounded, or wrong because the wrong chunks came back?

>Both RAG systems correctly stated that the appeal must be submitted within 14 calendar days of the decision letter using Form GR-305.


**5. What was missed, and was it a retrieval problem or a generation problem?** Quote the evidence.

>The LLM-only model missed the specific deadline and form because it did not have the relevant context. Retrieval enabled both RAG systems to give accurate, specific answers.

---
# Final questions

**1. What is the main difference between BM25 and embedding retrieval?**

>BM25 uses keyword matching and term frequency to rank documents. Embedding retrieval uses semantic similarity between vector representations of the question and document chunks. BM25 is good for exact terms, while embeddings can find relevant information even when different words are used.

**2. When can BM25 perform better than embeddings?**

>BM25 can perform better when a question contains exact keywords, form numbers, names, codes, or specific terms. It can match these terms directly without relying on semantic similarity.



**3. When can embeddings perform better than BM25?**

>Embeddings can perform better when a question is phrased differently from the source text or uses synonyms. They can identify semantically related information even when the exact keywords do not match.


**4. Why is cosine similarity used with embeddings?**

>Cosine similarity measures the angle between two vectors. It helps determine how semantically similar a question embedding is to a document embedding, regardless of their vector magnitudes.



**5. What information is actually sent to an LLM in a RAG system?**

>The LLM receives the user's question, the system instructions, and the retrieved text chunks. The chunks may include their IDs, titles, and text. The LLM uses this provided context to generate an answer.



**6. Does the LLM search the original documents itself?**

>No. In a standard RAG system, the retrieval component searches the document collection and selects relevant chunks. The LLM receives those chunks as part of its input and generates an answer from them.

**7. What happens if retrieval returns an irrelevant chunk?**

>The irrelevant chunk can distract the LLM, introduce misleading information, increase the number of input tokens, and reduce answer accuracy. The model may use incorrect context instead of the relevant information.


**8. Why can increasing Top-K sometimes improve RAG and sometimes make it worse?** Report K = 1, 3 and 8 on one question.

>On C-03, EmbeddingGemma RAG returned the correct GPA of 2.67 for all three values of K. With K = 1, it retrieved GO-02 and used 188 prompt tokens. With K = 3, it used 370 tokens; with K = 8, it used 880 tokens. Adding more chunks did not improve the answer in this experiment. K = 1 was sufficient and used the fewest tokens. Larger K can help when additional evidence is needed, but it can also introduce irrelevant or outdated information.

In [20]:

# Compare K = 1, 3 and 8 on C-03
test_item = next(q for q in qa_data if q["id"] == "C-03")

for k in [1, 3, 8]:
    retrieved = retrieve_embeddings(test_item["question"], k=k)
    messages = build_messages(test_item["question"], retrieved)

    response = client.chat.completions.create(
        model=MODEL_NAME,
        messages=messages
    )

    print(f"\n{'=' * 60}\nTop-K = {k}")
    print("Retrieved IDs:", [
        x["chunk"]["id"] for x in retrieved
    ])
    print("Answer:", response.choices[0].message.content)
    print("Prompt tokens:", response.usage.prompt_tokens)



Top-K = 1
Retrieved IDs: ['GO-02']
Answer: The minimum GPA needed is **2.67 on a 4.0 scale**. [1]
Prompt tokens: 188

Top-K = 3
Retrieved IDs: ['GO-02', 'GO-20', 'GO-05']
Answer: The minimum GPA needed is **2.67 on a 4.0 scale**. [1]
Prompt tokens: 370

Top-K = 8
Retrieved IDs: ['GO-02', 'GO-20', 'GO-05', 'GO-16', 'GO-01', 'GO-06', 'GO-19', 'GO-17']
Answer: The minimum GPA needed to qualify is **2.67 on the 4.0 scale**. [1]
Prompt tokens: 880


**9. Is a vector database required to build a RAG system?**

>No, a vector database isn’t necessary. On my laptop, the embeddings are stored in memory: in a NumPy array doc_embeddings in part B and in a tensor in part C. The search is done by calculating the cosine similarity between the question vector and the vectors of 74 fragments.

**10. Which retrieval approach worked better for this corpus, and what evidence supports that?** Give hit@3 for both over your ten queries and the C2 agreement counts for both RAG systems.

>EmbeddingGemma performed best in Part B, achieving hit@3 of 10/10 (100%), compared with BM25’s 6/10 (60%). It retrieved relevant chunks for Q-02, Q-04, Q-07 and Q-10, where BM25 failed. For example, on Q-04, BM25 ranked the archived 2025 rules first and missed GO-02, while EmbeddingGemma ranked the relevant 2026 eligibility rules in GO-02 first.

---
# Conclusion

>Next, I would build a Grant Office chatbot using EmbeddingGemma retrieval, because it achieved hit@3 of 10/10 compared with BM25’s 6/10 in my experiment. The chatbot would show sources and explain when the documents do not provide an answer, such as the grant amount for master’s students. I would also test a hybrid approach combining BM25 and embeddings to handle both exact form codes and questions in different languages. Before choosing it, I would compare its retrieval accuracy, answer quality and token usage with the current systems.
